In [9]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [10]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [11]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [12]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import optuna
import numpy as np

def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)
        maes = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # ✅ 计算 MAE
            mae = mean_absolute_error(y_val, y_pred)
            maes.append(mae)

        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    print(f'Best mean MAE: {study.best_value:.4f}')
    

In [13]:
# 数据预处理
df = pd.read_excel('../algae_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [14]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)



smiles_endpoint_combined = [f"{sm}_{ep}" for sm, ep in zip(new_smiles_list, endpoints)]


groups = smiles_endpoint_combined  # 可直接用于 GroupKFold




In [15]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [16]:
def xgb_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),   # L1 正则
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)  # L2 正则
    }
from xgboost import XGBRegressor

train_evaluate_regression_model_with_optuna(
    "XGBoost",
    XGBRegressor,
    xgb_param_func,
    X, y, groups
)

[I 2025-05-16 12:56:03,952] A new study created in memory with name: no-name-dafc7db2-1813-4497-a5de-99ede4ae15ad
Training XGBoost: 100%|██████████| 10/10 [02:03<00:00, 12.30s/it]
[I 2025-05-16 12:58:06,965] Trial 0 finished with value: 1.0966729729916458 and parameters: {'n_estimators': 396, 'max_depth': 10, 'learning_rate': 0.013186024959004587, 'subsample': 0.6835974371465126, 'colsample_bytree': 0.8291530641610648, 'reg_alpha': 0.7875349807436935, 'reg_lambda': 0.4791655965574563}. Best is trial 0 with value: 1.0966729729916458.
Training XGBoost: 100%|██████████| 10/10 [01:20<00:00,  8.01s/it]
[I 2025-05-16 12:59:27,074] Trial 1 finished with value: 1.0770543356016686 and parameters: {'n_estimators': 450, 'max_depth': 8, 'learning_rate': 0.021527965078609625, 'subsample': 0.7422841752525172, 'colsample_bytree': 0.9574641025708411, 'reg_alpha': 0.6284781333782696, 'reg_lambda': 0.6770085084561539}. Best is trial 1 with value: 1.0770543356016686.
Training XGBoost: 100%|██████████| 10

Best parameters for XGBoost: {'n_estimators': 600, 'max_depth': 14, 'learning_rate': 0.09366948490315667, 'subsample': 0.7198965937849887, 'colsample_bytree': 0.8391972522567769, 'reg_alpha': 0.5549324415159088, 'reg_lambda': 0.4572893892970914}
Best mean MAE: 0.8767


In [9]:
from lightgbm import LGBMRegressor

def lgbm_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1
    }

print("Training LightGBM (Poisson)...")
train_evaluate_regression_model_with_optuna(
    "LightGBM",
    lambda **params: LGBMRegressor(objective="poisson", **params),  # ✅ 加入 Poisson 目标
    lgbm_param_func,
    X, y, groups
)


[I 2025-05-15 22:07:13,545] A new study created in memory with name: no-name-03eeda18-e9b1-4efb-87f7-0588e5d0d784


Training LightGBM (Poisson)...


Training LightGBM: 100%|██████████| 10/10 [00:16<00:00,  1.68s/it]
[I 2025-05-15 22:07:30,326] Trial 0 finished with value: 1.020984853101984 and parameters: {'n_estimators': 167, 'max_depth': 12, 'num_leaves': 119, 'learning_rate': 0.2317946402764961, 'feature_fraction': 0.8227755465169926, 'bagging_fraction': 0.6161594714378685, 'bagging_freq': 1, 'reg_alpha': 0.6289742994585245, 'reg_lambda': 0.8524842363846702}. Best is trial 0 with value: 1.020984853101984.
Training LightGBM: 100%|██████████| 10/10 [00:50<00:00,  5.01s/it]
[I 2025-05-15 22:08:20,443] Trial 1 finished with value: 1.062058618505515 and parameters: {'n_estimators': 426, 'max_depth': 13, 'num_leaves': 219, 'learning_rate': 0.04893702343195991, 'feature_fraction': 0.7041133972221836, 'bagging_fraction': 0.7587072301366407, 'bagging_freq': 2, 'reg_alpha': 0.6534416286773325, 'reg_lambda': 0.839931640689567}. Best is trial 0 with value: 1.020984853101984.
Training LightGBM: 100%|██████████| 10/10 [01:01<00:00,  6.15s/it]

Best parameters for LightGBM: {'n_estimators': 537, 'max_depth': 17, 'num_leaves': 169, 'learning_rate': 0.2963042659500839, 'feature_fraction': 0.8046968165387371, 'bagging_fraction': 0.9136496662920279, 'bagging_freq': 3, 'reg_alpha': 0.031772557114264965, 'reg_lambda': 0.7576979131499533}
Best mean MAE: 0.9063


In [17]:
import random

# 固定随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # 设置固定种子

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import optuna
import numpy as np


class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_dnn_with_optuna_pytorch(X, y, groups, device=device):
    def dnn_param_func(trial):
        return {
            'hidden_layer_sizes': trial.suggest_categorical(
                'hidden_layer_sizes', [(50,), (100,), (150,), (100, 50), (150, 100, 50)]
            ),
            'activation': trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('solver', ['adam', 'sgd'])
        }

    def objective(trial):
        params = dnn_param_func(trial)
        model = DNNWithSoftplus(
            input_dim=X.shape[1],
            hidden_sizes=params['hidden_layer_sizes'],
            activation=params['activation']
        ).to(device)

        optimizer = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD
        }[params['optimizer']](model.parameters(), lr=params['learning_rate'], weight_decay=params['alpha'])

        loss_fn = nn.MSELoss()
        gkf = GroupKFold(n_splits=10)
        fold_maes = []

        for train_idx, val_idx in gkf.split(X, y, groups=groups):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
            train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

            model.train()
            for epoch in range(100):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                val_preds = model(torch.tensor(X_val).float().to(device)).cpu().numpy()
                mae = mean_absolute_error(y_val, val_preds)
                fold_maes.append(mae)

        return np.mean(fold_maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)
    print("\n✅ Best Parameters Found:")
    print(study.best_params)
    print(f"Mean MAE = {study.best_value:.4f}")
    return study.best_params


best_dnn_params = train_dnn_with_optuna_pytorch(X, y, groups)

[I 2025-05-16 14:05:58,925] A new study created in memory with name: no-name-b9956349-1fc2-424c-bef5-776fc217a0b6
[I 2025-05-16 14:07:46,265] Trial 0 finished with value: 0.9563126456191131 and parameters: {'hidden_layer_sizes': (150,), 'activation': 'tanh', 'alpha': 0.008739736644056877, 'learning_rate_init': 0.00013816470308548632, 'solver': 'sgd'}. Best is trial 0 with value: 0.9563126456191131.
[I 2025-05-16 14:09:45,364] Trial 1 finished with value: 0.8752749170401968 and parameters: {'hidden_layer_sizes': (150,), 'activation': 'logistic', 'alpha': 0.0005799349458084621, 'learning_rate_init': 0.003064084856610075, 'solver': 'sgd'}. Best is trial 1 with value: 0.8752749170401968.
[I 2025-05-16 14:11:44,331] Trial 2 finished with value: 0.8044252665692786 and parameters: {'hidden_layer_sizes': (50,), 'activation': 'relu', 'alpha': 0.00024802834876577767, 'learning_rate_init': 0.00030064860483241255, 'solver': 'sgd'}. Best is trial 2 with value: 0.8044252665692786.
[I 2025-05-16 14:1


✅ Best Parameters Found:
{'hidden_layer_sizes': (100,), 'activation': 'logistic', 'alpha': 1.0409123966878327e-05, 'learning_rate_init': 0.0005540921801893221, 'solver': 'adam'}
Mean MAE = 0.4589
